In [3]:
print("hello")

hello


In [2]:
import pyomo.environ as pyo

solver = pyo.SolverFactory("appsi_highs")
print(solver.available())  # Should print True

True


In [4]:
import pyomo.environ as pyo

# INPUT DATA — replace these with your own values
solar = {
    10: 1.5,  # Available solar power in kW
    11: 3.0,
    12: 2.0,
}

appliance_power = {
    "washing_machine": 1.0,  # Power requirement in kW
    "water_pump": 0.8,
    "water_heater": 2.0,
}

# BUILD THE MODEL
model = pyo.ConcreteModel()

model.hours = pyo.Set(initialize=list(solar))
model.appliances = pyo.Set(initialize=list(appliance_power))

# 1 = appliance runs in this slot; 0 = it does not
model.on = pyo.Var(
    model.appliances,
    model.hours,
    domain=pyo.Binary,
)

# Each appliance must run in exactly one slot
def run_once_rule(m, appliance):
    return sum(m.on[appliance, h] for h in m.hours) == 1

model.run_once = pyo.Constraint(
    model.appliances,
    rule=run_once_rule,
)

# Combined appliance power cannot exceed available solar
def solar_limit_rule(m, hour):
    return sum(
        appliance_power[a] * m.on[a, hour]
        for a in m.appliances
    ) <= solar[hour]

model.solar_limit = pyo.Constraint(
    model.hours,
    rule=solar_limit_rule,
)

# For now, we only need a feasible schedule.
# A constant objective means every feasible schedule is acceptable.
model.objective = pyo.Objective(expr=0)

# SOLVE
solver = pyo.SolverFactory("appsi_highs")
results = solver.solve(model)

# DISPLAY RESULTS
if results.solver.termination_condition == pyo.TerminationCondition.optimal:
    for h in model.hours:
        running = [
            a for a in model.appliances
            if pyo.value(model.on[a, h]) > 0.5
        ]

        total_power = sum(appliance_power[a] for a in running)

        print(f"{h}:00 — {', '.join(running) or 'No appliances'}")
        print(f"  Appliance power: {total_power:.1f} kW")
        print(f"  Available solar: {solar[h]:.1f} kW")
else:
    print("Solver outcome:", results.solver.termination_condition)

10:00 — water_pump
  Appliance power: 0.8 kW
  Available solar: 1.5 kW
11:00 — washing_machine
  Appliance power: 1.0 kW
  Available solar: 3.0 kW
12:00 — water_heater
  Appliance power: 2.0 kW
  Available solar: 2.0 kW


In [3]:
import pyomo.environ as pyo

# Available solar power for each one-hour slot, in kW
solar = {
    8: 0.5,
    9: 1.0,
    10: 1.5,
    11: 3.0,
    12: 3.5,
    13: 3.2,
    14: 2.5,
    15: 1.8,
    16: 1.0,
    17: 0.3,
}

# Appliance power requirements, in kW
appliance_power = {
    "washing_machine": 1.0,
    "water_pump": 0.8,
    "water_heater": 2.0,
    "dishwasher": 1.2,
    "rice_cooker": 0.7,
    "vacuum_cleaner": 0.9,
    "electric_iron": 1.5,
    "ev_charger": 3.0,
}
# BUILD THE MODEL
model = pyo.ConcreteModel()

model.hours = pyo.Set(initialize=list(solar))
model.appliances = pyo.Set(initialize=list(appliance_power))

# 1 = appliance runs in this slot; 0 = it does not
model.on = pyo.Var(
    model.appliances,
    model.hours,
    domain=pyo.Binary,
)

# Each appliance must run in exactly one slot
def run_once_rule(m, appliance):
    return sum(m.on[appliance, h] for h in m.hours) == 1

model.run_once = pyo.Constraint(
    model.appliances,
    rule=run_once_rule,
)

# Combined appliance power cannot exceed available solar
def solar_limit_rule(m, hour):
    return sum(
        appliance_power[a] * m.on[a, hour]
        for a in m.appliances
    ) <= solar[hour]

model.solar_limit = pyo.Constraint(
    model.hours,
    rule=solar_limit_rule,
)

# For now, we only need a feasible schedule.
# A constant objective means every feasible schedule is acceptable.
model.objective = pyo.Objective(expr=0)

# SOLVE
solver = pyo.SolverFactory("appsi_highs")
results = solver.solve(model)

# DISPLAY RESULTS
if results.solver.termination_condition == pyo.TerminationCondition.optimal:
    for h in model.hours:
        running = [
            a for a in model.appliances
            if pyo.value(model.on[a, h]) > 0.5
        ]

        total_power = sum(appliance_power[a] for a in running)

        print(f"{h}:00 — {', '.join(running) or 'No appliances'}")
        print(f"  Appliance power: {total_power:.1f} kW")
        print(f"  Available solar: {solar[h]:.1f} kW")
else:
    print("Solver outcome:", results.solver.termination_condition)

8:00 — No appliances
  Appliance power: 0.0 kW
  Available solar: 0.5 kW
9:00 — No appliances
  Appliance power: 0.0 kW
  Available solar: 1.0 kW
10:00 — No appliances
  Appliance power: 0.0 kW
  Available solar: 1.5 kW
11:00 — ev_charger
  Appliance power: 3.0 kW
  Available solar: 3.0 kW
12:00 — washing_machine, water_pump, rice_cooker
  Appliance power: 2.5 kW
  Available solar: 3.5 kW
13:00 — water_heater, dishwasher
  Appliance power: 3.2 kW
  Available solar: 3.2 kW
14:00 — No appliances
  Appliance power: 0.0 kW
  Available solar: 2.5 kW
15:00 — electric_iron
  Appliance power: 1.5 kW
  Available solar: 1.8 kW
16:00 — vacuum_cleaner
  Appliance power: 0.9 kW
  Available solar: 1.0 kW
17:00 — No appliances
  Appliance power: 0.0 kW
  Available solar: 0.3 kW


In [5]:
import pyomo.environ as pyo

# Available solar power for each one-hour slot, in kW
solar = {
    8: 0.5,
    9: 1.0,
    10: 1.5,
    11: 3.0,
    12: 3.5,
    13: 3.2,
    14: 2.5,
    15: 1.8,
    16: 1.0,
    17: 0.3,
}

# Appliance power requirements, in kW
appliance_power = {
    "washing_machine": 1.0,
    "water_pump": 0.8,
    "water_heater": 2.0,
    "dishwasher": 1.2,
    "rice_cooker": 0.7,
    "vacuum_cleaner": 0.9,
    "electric_iron": 1.5,
    "ev_charger": 3.0,
}

# BUILD THE MODEL
model = pyo.ConcreteModel()

model.hours = pyo.Set(initialize=sorted(solar), ordered=True)
model.appliances = pyo.Set(initialize=list(appliance_power))

# 1 = appliance runs during this hour; 0 = it does not
model.on = pyo.Var(
    model.appliances,
    model.hours,
    domain=pyo.Binary,
)

# Each appliance must run for exactly one hour
def run_once_rule(m, appliance):
    return sum(m.on[appliance, h] for h in m.hours) == 1


model.run_once = pyo.Constraint(
    model.appliances,
    rule=run_once_rule,
)

# Total appliance power used during each hour
def total_power_rule(m, hour):
    return sum(
        appliance_power[a] * m.on[a, hour]
        for a in m.appliances
    )


model.total_power = pyo.Expression(
    model.hours,
    rule=total_power_rule,
)

# Appliance demand cannot exceed available solar power
def solar_limit_rule(m, hour):
    return m.total_power[hour] <= solar[hour]


model.solar_limit = pyo.Constraint(
    model.hours,
    rule=solar_limit_rule,
)

# Prefer earlier operating hours:
# minimize the sum of all appliance start times
model.objective = pyo.Objective(
    expr=sum(
        h * model.on[a, h]
        for a in model.appliances
        for h in model.hours
    ),
    sense=pyo.minimize,
)

# SOLVE
solver = pyo.SolverFactory("appsi_highs")

if not solver.available(exception_flag=False):
    raise RuntimeError(
        "HiGHS is unavailable. Install it with: uv pip install highspy"
    )

# Read the solution only after checking the solver outcome
results = solver.solve(model, load_solutions=False)
termination = results.solver.termination_condition

if termination == pyo.TerminationCondition.optimal:
    model.solutions.load_from(results)

    print("Optimal schedule found\n")

    total_energy_used = 0.0
    total_solar_energy = 0.0
    slot_duration_hours = 1.0

    for h in model.hours:
        running = [
            a for a in model.appliances
            if pyo.value(model.on[a, h]) > 0.5
        ]

        used_power = pyo.value(model.total_power[h])
        unused_power = max(0.0, solar[h] - used_power)

        total_energy_used += used_power * slot_duration_hours
        total_solar_energy += solar[h] * slot_duration_hours

        names = ", ".join(running) if running else "No appliances"

        print(f"{h:02d}:00–{h + 1:02d}:00 — {names}")

        for appliance in running:
            print(f"  {appliance}: {appliance_power[appliance]:.1f} kW")

        print(f"  Total appliance power: {used_power:.1f} kW")
        print(f"  Available solar:       {solar[h]:.1f} kW")
        print(f"  Unused solar:          {unused_power:.1f} kW")
        print()

    print(f"Total solar energy available: {total_solar_energy:.1f} kWh")
    print(f"Total appliance energy used:  {total_energy_used:.1f} kWh")
    print(
        f"Total unused solar energy:    "
        f"{total_solar_energy - total_energy_used:.1f} kWh"
    )

elif termination == pyo.TerminationCondition.infeasible:
    print(
        "No feasible schedule exists. The appliances cannot all run "
        "for one hour within the available solar power limits."
    )

else:
    print("Solver outcome:", termination)

Optimal schedule found

08:00–09:00 — No appliances
  Total appliance power: 0.0 kW
  Available solar:       0.5 kW
  Unused solar:          0.5 kW

09:00–10:00 — washing_machine
  washing_machine: 1.0 kW
  Total appliance power: 1.0 kW
  Available solar:       1.0 kW
  Unused solar:          0.0 kW

10:00–11:00 — water_pump, rice_cooker
  water_pump: 0.8 kW
  rice_cooker: 0.7 kW
  Total appliance power: 1.5 kW
  Available solar:       1.5 kW
  Unused solar:          0.0 kW

11:00–12:00 — water_heater, vacuum_cleaner
  water_heater: 2.0 kW
  vacuum_cleaner: 0.9 kW
  Total appliance power: 2.9 kW
  Available solar:       3.0 kW
  Unused solar:          0.1 kW

12:00–13:00 — dishwasher, electric_iron
  dishwasher: 1.2 kW
  electric_iron: 1.5 kW
  Total appliance power: 2.7 kW
  Available solar:       3.5 kW
  Unused solar:          0.8 kW

13:00–14:00 — ev_charger
  ev_charger: 3.0 kW
  Total appliance power: 3.0 kW
  Available solar:       3.2 kW
  Unused solar:          0.2 kW

14:00–1